# Project 5: LoRA supervised fine-tuning for candle-data repair instructions

**Name:** To be completed  
**Course:** CSCI 4820 — section to be completed

`candleprep` downloads the original CSV from the public NLP-proj5 repository,
creates corruptions, and verifies their JSON repair labels offline. This notebook
displays the data and adapts the starter's Mistral/PEFT `Trainer` workflow.
The model receives only corrupted CSV and predicts a repair plan, never the
original values, corruption history, or reconstructed data.

Use a CUDA-enabled PyTorch environment with Transformers installed. The setup
cell installs PEFT 0.21.3, bitsandbytes 0.50.2, and Accelerate 1.15.0 alongside
candleprep. These versions were verified locally with Transformers 5.8.1.
After the first installation, restart the kernel and run from the top.
Run from top to bottom. The first source/model downloads need internet access;
later runs reuse local caches. Training can take hours; extra generated-answer
comparison is bounded and optional.

## 1. Install the CPU preparation package

Install the data/display package and the LoRA/4-bit training dependencies in
the active kernel. No separate CSV download or repository clone is required.
After installation, restart the kernel once before continuing.

Math rendering is disabled for styled data tables so currency signs remain
literal text. This changes presentation only, not model inputs or labels.


In [ ]:
%pip install -q "candleprep[notebook]==0.2.0" "peft==0.21.3" "accelerate==1.15.0" "bitsandbytes==0.50.2"
from candleprep import CandleDataset
import pandas as pd
pd.set_option("styler.html.mathjax", False)  # Display currency signs as literal text


## 2. Prepare the dataset

One call downloads the pinned original CSV, verifies its checksum, and creates
10,000 training, 1,000 validation, and 2,000 test pairs. Existing matching files
are verified and reused. Source windows are split before corruption to avoid
leakage. All labels are produced by deterministic tools, without model inference.
Section 4 selects how many of the prepared training pairs to use for SFT.


In [ ]:
dataset = CandleDataset.prepare(output_dir="generated_data/v1", seed=4820)
dataset.summary()


## 3. Inspect the data and the actual SFT labels

Display the first, middle, and last five rows of the original and synthetic data.
The label table shows three recoverable examples and two whole-window omissions.
Its reconstruction is evidence that the JSON label is correct. The supervised
target remains the JSON repair instructions, not reconstructed CSV.


In [ ]:
dataset.display_samples(rows=5)
dataset.display_labels(rows=5)
dataset.display_coverage()


## 4. Model imports and experiment settings

Set `TRAINING_EXAMPLES` to the number of random training pairs to use
(default 1,000; `None` uses all 10,000). Sampling uses `SEED`, preserves
each complete CSV window and JSON label, and never draws from validation/test.
Only selected pairs are tokenized for training. Change `RESULTS` for a new
training configuration; saved sample IDs prevent mixing checkpoints.
After changing training settings, rerun Sections 4–9, including tokenization.

`DEMO_EXAMPLES` controls the same before/after validation cases in Sections
6 and 9 (default 5; range 1–1,000). Change it and rerun Section 4, then
Sections 6, 9, and 10 to regenerate/reuse demos and charts without retraining.
`COMPARISON_EXAMPLES` controls only Section 11: an additional test sample
per model, with 0 disabling it. It does not select training or demo cases.

Retain the starter's 4-bit Mistral and LoRA approach. Batch size 1, accumulation
16, and gradient checkpointing preserve the tested low-memory configuration.
The previous rank-16 run peaked at 5.90 GiB allocated / 6.17 GiB reserved by
PyTorch; whole-device peak was not logged, so a 10-GB GPU still needs verification.

After an interruption stops, call `free_gpu_resources()` in a new cell. It clears
live model references and traceback/output caches; saved checkpoints remain.
After a kernel restart, rerun setup/definition cells before running later cells.
Saved outputs alone do not restore Python variables.


In [ ]:
import gc, hashlib, json, math, random, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from candleprep.task import TASK, user_text
from candleprep.evaluation import score_predictions

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
RESULTS = Path("candle_results/sampled_sft_v1")  # New directory for changed training settings
SEED = 4820
LORA_RANK = 16  # Primary adapter used for the before/after comparison
TRAINING_EXAMPLES = 1000  # Seeded random pairs; None uses all 10,000
DEMO_EXAMPLES = 5  # Matched before/after validation cases; independent of training
EPOCHS = 1
BATCH_SIZE = 1
ACCUMULATION = 16
LEARNING_RATE = 0.0002
MAX_LENGTH = 1536
MAX_NEW_TOKENS = 128
VALIDATION_EXAMPLES = 64
VALIDATION_EVERY = 25
COMPARISON_EXAMPLES = 0  # Optional generation: set to 25 for 25 cases per model
RESULTS.mkdir(parents=True, exist_ok=True)

def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")

training_records = dataset.records("train")
if TRAINING_EXAMPLES is not None:
    if type(TRAINING_EXAMPLES) is not int or not 1 <= TRAINING_EXAMPLES <= len(training_records):
        raise ValueError(f"TRAINING_EXAMPLES must be None or 1..{len(training_records)}")
    training_records = random.Random(SEED).sample(training_records, TRAINING_EXAMPLES)
print(f"Training on {len(training_records):,} cached input/label pairs.")

settings = {"model": MODEL_ID, "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
            "accumulation": ACCUMULATION, "learning_rate": LEARNING_RATE,
            "max_length": MAX_LENGTH, "max_new_tokens": MAX_NEW_TOKENS,
            "task_sha256": hashlib.sha256(TASK.encode()).hexdigest(),
            "dataset_artifacts": dataset.manifest["artifact_sha256"],
            "training_ids": [r["id"] for r in training_records]}
configuration = RESULTS / "settings.json"
if configuration.exists() and json.loads(configuration.read_text()) != settings:
    raise ValueError("Run settings changed; choose a new RESULTS directory")
save_json(configuration, settings)


### Manual GPU cleanup

This cell defines and runs cleanup. Rerun it after an interrupted run has
stopped, or call `free_gpu_resources()` in a new cell. Saved checkpoints
remain available. The printed figures are for the current PyTorch kernel;
other applications and the CUDA context can still occupy device memory.


In [ ]:
def release_gpu():
    """Release optimizer/module caches and CUDA workspaces after model disposal."""
    import sys
    import torch
    bnb = sys.modules.get("bitsandbytes")
    if bnb is not None and hasattr(bnb, "optim"):
        bnb.optim.GlobalOptimManager.get_instance().initialize()
    gc.collect()
    gc.collect()
    if torch.cuda.is_initialized():
        torch.cuda.synchronize()
        clear_workspaces = getattr(getattr(torch, "_C", None), "_cuda_clearCublasWorkspaces", None)
        if clear_workspaces is not None:
            clear_workspaces()
        torch.cuda.empty_cache()

def free_gpu_resources():
    """Call in a new cell after an interrupted run has stopped."""
    import sys
    import traceback
    import torch

    # Drop this notebook's live models/training objects; keep data and checkpoints.
    for name in ("trainer", "model", "base_model", "optimizer", "scheduler"):
        globals().pop(name, None)

    # An interrupted Trainer/generate call can survive through traceback locals.
    for name in ("last_traceback", "last_value", "last_exc"):
        value = getattr(sys, name, None)
        tb = value if name == "last_traceback" else getattr(value, "__traceback__", None)
        if tb is not None:
            traceback.clear_frames(tb)
        if hasattr(sys, name):
            setattr(sys, name, None)
    shell = globals().get("get_ipython", lambda: None)()
    if shell is not None:
        shell.run_line_magic("reset", "-f out")  # Clear cached Python results.
        shell.last_execution_result = None
        tb = getattr(shell.InteractiveTB, "tb", None)
        if tb is not None:
            traceback.clear_frames(tb)
            shell.InteractiveTB.tb = None
    try:
        release_gpu()
        if not torch.cuda.is_initialized():
            print("CUDA is not initialized; Python GPU references cleared.")
            return
        print(f"GPU cleanup: allocated {torch.cuda.memory_allocated()/1024**3:.2f} GiB; "
              f"reserved {torch.cuda.memory_reserved()/1024**3:.2f} GiB.")
    except RuntimeError as error:
        print(f"CUDA cleanup failed: {error}. Restart the kernel before loading another model.")

# Running this cell performs the cleanup immediately.
free_gpu_resources()


## 5. Tokenization and small model helpers

Keep the complete JSON target. Mask prompt/padding tokens with `-100` so training
loss applies only to the answer. Prediction decodes only newly generated tokens.
`predict()` is used for evaluation; Trainer consumes cached labels directly.
The progress bar reports examples/ETA, and Trainer supplies its own step/loss bar.

Validation uses a fixed sample and target-only teacher-forced loss/accuracy.
Only argmax token IDs are retained for metrics, avoiding full vocabulary
logit storage. It does not call model.generate().


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def prompt_tokens(tokenizer, record):
    return tokenizer.apply_chat_template([{"role": "user", "content": user_text(record)}],
        tokenize=True, add_generation_prompt=True, return_dict=False)

def tokenize_record(record, tokenizer):
    messages = [{"role": "user", "content": user_text(record)}]
    prompt = prompt_tokens(tokenizer, record)
    full = tokenizer.apply_chat_template(messages + [{"role": "assistant", "content":
        json.dumps(record["target"], separators=(",", ":"))}], tokenize=True, return_dict=False)
    if full[:len(prompt)] != prompt or len(full) > MAX_LENGTH:
        raise ValueError("Prompt boundary or complete target exceeds the token budget")
    return {"input_ids": full, "attention_mask": [1] * len(full),
            "labels": [-100] * len(prompt) + full[len(prompt):]}

train_dataset = Dataset.from_list([tokenize_record(r, tokenizer) for r in training_records])
tokenized_training_ids = [r["id"] for r in training_records]
print(train_dataset)

def load_base():
    if not torch.cuda.is_available():
        raise RuntimeError("The model stages require a CUDA GPU")
    free, _ = torch.cuda.mem_get_info()
    print(f"Loading quantized Mistral; {free/1024**3:.2f} GiB GPU memory free.", flush=True)
    set_seed(SEED)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=config,
                                               device_map={"": 0}, attn_implementation="sdpa")


def predict(model, records, label="Evaluation"):
    model.eval()
    model.config.use_cache = True
    predictions = []
    for record in tqdm(records, desc=label, unit="example"):
        ids = prompt_tokens(tokenizer, record)
        if len(ids) + MAX_NEW_TOKENS > MAX_LENGTH:
            raise ValueError("Complete prompt exceeds the generation budget")
        inputs = torch.tensor([ids], device=model.get_input_embeddings().weight.device)
        with torch.inference_mode():
            output = model.generate(input_ids=inputs, attention_mask=torch.ones_like(inputs),
                max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        predictions.append(tokenizer.decode(output[0, len(ids):], skip_special_tokens=True).strip())
    return predictions

validation_records = random.Random(SEED).sample(dataset.records("validation"), VALIDATION_EXAMPLES)
validation_dataset = Dataset.from_list([tokenize_record(r, tokenizer) for r in validation_records])

def validation_token_ids(logits, labels):
    return (logits[0] if isinstance(logits, tuple) else logits).argmax(dim=-1)

def validation_metrics(batch):
    predictions, labels = batch
    mask = labels[:, 1:] != -100
    correct = (predictions[:, :-1] == labels[:, 1:]) & mask
    return {"target_token_accuracy": float(correct.sum() / mask.sum())}


## 6. Configurable examples before SFT

Freeze `DEMO_EXAMPLES` validation cases before generation. With at least
three cases, the selection includes positive repairs and an unrecoverable
window. Display the complete task/input, model answer, and expected JSON
separately. Baseline responses are cached by count and reused on reruns.
This cell generates only the requested demos before training.


In [ ]:
demo_records = dataset.examples(count=DEMO_EXAMPLES)
before_path = RESULTS / f"before_{DEMO_EXAMPLES}.json"
if before_path.exists():
    frozen = json.loads(before_path.read_text())
    assert frozen["ids"] == [r["id"] for r in demo_records]
    assert frozen["prompts"] == [user_text(r) for r in demo_records]
    before_predictions = frozen["predictions"]
else:
    model = load_base()
    try:
        before_predictions = predict(model, demo_records, f"Before SFT: {DEMO_EXAMPLES} examples")
        save_json(before_path, {"ids": [r["id"] for r in demo_records],
            "prompts": [user_text(r) for r in demo_records], "predictions": before_predictions})
    finally:
        del model
        release_gpu()
save_json(RESULTS / f"before_{DEMO_EXAMPLES}_metrics.json", score_predictions(before_predictions, demo_records))
dataset.display_predictions(demo_records, before_predictions)


## 7. Starter-style LoRA training

The following function keeps the starter's LoRA configuration, TrainingArguments,
Trainer, and adapter save visible. It trains on the Section 4 sample:
1,000 pairs require about one tenth the training work of all 10,000.
Prompt loss is masked; training never calls `generate()`.
Checkpoints are saved every 100 steps. Matching completed ranks are reused;
interrupted training resumes from the latest checkpoint.

Trainer also logs validation loss and target-token accuracy on the configured
small validation sample. These measurements do not update model weights.


In [ ]:
def train_adapter(rank):
    destination = RESULTS / f"rank_{rank}"
    adapter = destination / "adapter"
    rank_settings = {**settings, "rank": rank, "lora_alpha": 32, "lora_dropout": 0.05,
                     "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}
    config_path = destination / "settings.json"
    if config_path.exists() and json.loads(config_path.read_text()) != rank_settings:
        raise ValueError("Rank settings changed; choose a new RESULTS directory")
    if (destination / "training.json").exists() and (adapter / "adapter_model.safetensors").exists():
        print("Reusing saved adapter:", adapter)
        return adapter
    if tokenized_training_ids != settings["training_ids"]:
        raise ValueError("Training sample changed; rerun Section 5 before training")
    destination.mkdir(parents=True, exist_ok=True)
    save_json(config_path, rank_settings)
    checkpoint = get_last_checkpoint(str(destination))
    model = trainer = None
    try:
        model = prepare_model_for_kbit_training(load_base())
        model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=32, lora_dropout=0.05,
            target_modules=rank_settings["target_modules"], bias="none", task_type="CAUSAL_LM"))
        model.config.use_cache = False
        model.print_trainable_parameters()
        arguments = TrainingArguments(output_dir=str(destination),
            per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=ACCUMULATION,
            num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
            gradient_checkpointing=True, logging_steps=5, logging_first_step=True, disable_tqdm=False,
            save_steps=100, save_total_limit=2, report_to="none", seed=SEED, data_seed=SEED,
            remove_unused_columns=False, eval_strategy="steps", eval_steps=VALIDATION_EVERY,
            per_device_eval_batch_size=1, eval_accumulation_steps=1)
        trainer = Trainer(model=model, args=arguments, train_dataset=train_dataset,
            eval_dataset=validation_dataset, compute_metrics=validation_metrics,
            preprocess_logits_for_metrics=validation_token_ids,
            processing_class=tokenizer, data_collator=DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, label_pad_token_id=-100))
        torch.cuda.reset_peak_memory_stats()
        started = time.monotonic()
        result = trainer.train(resume_from_checkpoint=checkpoint)
        if not math.isfinite(result.training_loss):
            raise RuntimeError("Training loss is not finite")
        validation = trainer.evaluate()  # Final teacher-forced validation point; no generate().
        trainer.model.save_pretrained(adapter)
        tokenizer.save_pretrained(adapter)
        trainer.state.save_to_json(str(destination / "trainer_state.json"))
        save_json(destination / "training.json", {"rank": rank, "examples": len(train_dataset),
            "steps": trainer.state.global_step, "epochs": trainer.state.epoch,
            "validation": validation, "validation_examples": len(validation_dataset),
            "loss": result.training_loss, "seconds": time.monotonic() - started,
            "peak_allocated_gib": torch.cuda.max_memory_allocated()/1024**3,
            "peak_reserved_gib": torch.cuda.max_memory_reserved()/1024**3})
        return adapter
    finally:
        del trainer, model
        release_gpu()


## 8. Train the primary adapter

This cell trains directly from the cached tokenized input/JSON-label pairs.
No model responses or recovery tools are used to create labels here.


In [ ]:
assert before_path.exists(), "Run the configured demo baseline first"
selected_adapter = train_adapter(LORA_RANK)
print("Saved primary adapter:", selected_adapter)


## 9. The same configured examples after SFT

Reload the saved adapter onto a fresh base and use the same configured IDs, prompts,
and generation settings. This small comparison illustrates behavior; the optional
sampled comparison below can broaden the generated-answer check. After a kernel restart,
rerun setup through Section 5, Section 6 (it reuses the baseline), and the
Section 7 function definition. You can then run this cell without retraining.


In [ ]:
frozen = json.loads((RESULTS / f"before_{DEMO_EXAMPLES}.json").read_text())
demo_records = dataset.examples(count=DEMO_EXAMPLES)
assert frozen["ids"] == [r["id"] for r in demo_records]
assert frozen["prompts"] == [user_text(r) for r in demo_records]
selected_adapter = RESULTS / f"rank_{LORA_RANK}" / "adapter"
model = PeftModel.from_pretrained(load_base(), selected_adapter)
try:
    after_predictions = predict(model, demo_records, f"After SFT: {DEMO_EXAMPLES} examples")
    dataset.display_predictions(demo_records, after_predictions, frozen["predictions"])
    save_json(selected_adapter.parent / f"after_{DEMO_EXAMPLES}.json", {**frozen, "predictions": after_predictions})
    save_json(selected_adapter.parent / f"after_{DEMO_EXAMPLES}_metrics.json", score_predictions(after_predictions, demo_records))
finally:
    del model
    release_gpu()


## 10. Training results and charts — no generation required

Read Trainer's saved history to plot training loss, validation loss, and
**teacher-forced validation target-token accuracy**. New training runs evaluate
`VALIDATION_EXAMPLES` fixed validation cases every `VALIDATION_EVERY` steps
and at the end (defaults: 64 cases, 25 steps). These are direct
forward passes against cached labels, not autoregressive response generation.
Prompt/padding tokens are excluded and causal predictions are shifted correctly.

The configured demo table and bar chart report generated-answer metrics
separately. Token accuracy is not complete JSON-plan accuracy. These charts do
not depend on the optional sample below and can be run as soon as training is
saved. Older runs without validation history show the available training curve;
missing validation metrics are never invented and saved adapters are not retrained.

For the assignment's rank experiment, change LORA_RANK and rerun Sections 1–9.
Each rank is saved separately. Use another RESULTS directory when changing
training hyperparameters such as learning rate or epoch count, and document the
values/observations in Markdown. There is no automatic multi-rank training loop.


In [ ]:
rank_dir = RESULTS / f"rank_{LORA_RANK}"
state_path = rank_dir / "trainer_state.json"
if state_path.exists():
    history = pd.DataFrame(json.loads(state_path.read_text())["log_history"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for key, label in (("loss", "Training loss"), ("eval_loss", "Validation loss")):
        if key in history:
            history.dropna(subset=[key]).plot(x="step", y=key, ax=axes[0], label=label)
    key = "eval_target_token_accuracy"
    if key in history:
        history.dropna(subset=[key]).plot(x="step", y=key, ax=axes[1], legend=False)
    else:
        axes[1].set_axis_off()
        axes[1].text(0.05, 0.5, "Validation metrics were not recorded\nfor this older run.", transform=axes[1].transAxes)
    axes[0].set(title=f"LoRA rank {LORA_RANK}: loss", xlabel="Optimizer step", ylabel="Cross-entropy loss")
    axes[1].set(title="Teacher-forced validation target-token accuracy", xlabel="Optimizer step", ylabel="Accuracy", ylim=(0, 1))
    plt.tight_layout()
    plt.show()
else:
    print("No saved Trainer history yet:", state_path)

before_metrics = RESULTS / f"before_{DEMO_EXAMPLES}_metrics.json"
if before_metrics.exists():
    reports = {f"Before SFT ({DEMO_EXAMPLES} cases)": json.loads(before_metrics.read_text())}
    after_path = rank_dir / f"after_{DEMO_EXAMPLES}_metrics.json"
    if after_path.exists():
        reports[f"After SFT rank {LORA_RANK} ({DEMO_EXAMPLES} cases)"] = json.loads(after_path.read_text())
    metrics = ["recoverability_accuracy", "operation_sequence_accuracy", "full_answer_accuracy", "format_validity"]
    table = pd.DataFrame({label: {key: report[key] for key in metrics} for label, report in reports.items()})
    display(table)
    table.T.plot.bar(ylim=(0, 1), figsize=(10, 4), title=f"{DEMO_EXAMPLES}-case generated-answer comparison")
    plt.tight_layout()
    plt.show()


## 11. Optional small generated-answer comparison

Set COMPARISON_EXAMPLES to 25 for 25 fixed held-out cases per model; 0 skips this
section. Both original base and trained adapter receive the same sampled inputs.
The progress bars show the actual count. Results are saved per rank and sample
size. This small-sample estimate does not claim accuracy over all 2,000 test cases.
Training curves and demo results above are already available independently.

Sources: source CSV in [NLP-proj5](https://github.com/Absolute-Convergence/NLP-proj5),
the supplied local starter, [PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization),
[Trainer](https://huggingface.co/docs/transformers/main_classes/trainer), and
[chat templates](https://huggingface.co/docs/transformers/chat_templating).
AI/source assistance is summarized in the separate local `proj5doc.md`.


In [ ]:
if COMPARISON_EXAMPLES > 0:
    comparison_records = random.Random(SEED).sample(dataset.records("test"), COMPARISON_EXAMPLES)
    sampled_reports = {}
    for label in ("base", "selected"):
        model = load_base()
        try:
            if label == "selected":
                model = PeftModel.from_pretrained(model, RESULTS / f"rank_{LORA_RANK}" / "adapter")
            predictions = predict(model, comparison_records, f"{label}: {len(comparison_records)} test examples")
            report = score_predictions(predictions, comparison_records)
            sampled_reports[label] = {"ids": [r["id"] for r in comparison_records], "predictions": predictions, "metrics": report}
        finally:
            del model
            release_gpu()
    save_json(RESULTS / f"rank_{LORA_RANK}" / f"sample_{COMPARISON_EXAMPLES}.json", sampled_reports)
    display(pd.DataFrame({label: result["metrics"] for label, result in sampled_reports.items()}).drop(index="confusion_matrix"))
else:
    print("Extra generation skipped. Set COMPARISON_EXAMPLES to e.g. 25 to enable it.")
